# Tactical Fingerprints — Full Analysis

**Single notebook. Run All. Everything saved back to Drive.**

Covers:
- Drive mount + repo setup
- Exploratory Data Analysis (event distributions, team stats, competition comparisons)
- Passing network construction + visualisation
- Feature extraction + correlation analysis
- PCA + UMAP + hierarchical clustering
- Tactical Fingerprint interpretation
- Outcome analysis (win rate, points, goal difference by fingerprint)
- Cross-competition fingerprint comparison
- All outputs saved to Drive

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Configure ────────────────────────────────────────────────────────────────
DRIVE_DATA   = '/content/drive/MyDrive/Football_Events_SDS/Data'
DRIVE_OUT    = '/content/drive/MyDrive/Football_Events_SDS/Results'
REPO_DIR     = '/content/Football-Analytics'

COMPETITIONS = [
    'England', 'Spain', 'Italy', 'France', 'Germany',
    'European_Championship', 'World_Cup'
]
N_CLUSTERS   = 5   # adjust after inspecting the silhouette curve below

os.environ['WYSCOUT_RAW_DIR'] = DRIVE_DATA
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yasarkerem/Football-Analytics.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!pip install -q -r requirements.txt
print('Ready')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 5)})
sns.set_theme(style='whitegrid', palette='tab10')

OUT = Path(DRIVE_OUT)
print('Imports OK')

---
## 1. Load Data

In [ ]:
from src.data.loader import load_events, load_matches, load_teams, load_players, load_playerank

events_df   = load_events(COMPETITIONS)
matches_df  = load_matches(COMPETITIONS)
teams_df    = load_teams()
players_df  = load_players()
playerank   = load_playerank()

print(f'Events:    {len(events_df):>10,}')
print(f'Matches:   {len(matches_df):>10,}')
print(f'Teams:     {len(teams_df):>10,}')
print(f'Players:   {len(players_df):>10,}')
print(f'PlayerAnk: {len(playerank):>10,}')

---
## 2. Exploratory Data Analysis

In [ ]:
# ── Event type distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

events_df['eventName'].value_counts().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Event Type Distribution (all competitions)')
axes[0].set_xlabel('Count')

events_df.groupby('competition')['eventName'].count().sort_values().plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Events per Competition')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig(OUT / 'eda_event_distribution.png', dpi=130)
plt.show()

In [ ]:
# ── Spatial heatmap — where events happen on the pitch ────────────────────────
fig, axes = plt.subplots(1, len(COMPETITIONS), figsize=(4*len(COMPETITIONS), 4))
if len(COMPETITIONS) == 1: axes = [axes]

for ax, comp in zip(axes, COMPETITIONS):
    sub = events_df[events_df['competition'] == comp]
    ax.hist2d(sub['pos_x'].dropna(), sub['pos_y'].dropna(),
              bins=30, cmap='hot_r', density=True)
    ax.set_title(comp, fontsize=8)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_aspect('equal')
    ax.axvline(33.3, color='white', lw=0.5, alpha=0.6)
    ax.axvline(66.6, color='white', lw=0.5, alpha=0.6)

plt.suptitle('Event Density Heatmap by Competition (x=attacking direction)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(OUT / 'eda_spatial_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Match outcomes distribution ───────────────────────────────────────────────
home_goals = matches_df['homeScore'].dropna()
away_goals = matches_df['awayScore'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(home_goals, bins=range(0, 11), alpha=0.7, color='steelblue', label='Home')
axes[0].hist(away_goals, bins=range(0, 11), alpha=0.7, color='coral', label='Away')
axes[0].set_title('Goals Scored Distribution')
axes[0].legend()

total_goals = home_goals + away_goals
axes[1].hist(total_goals.dropna(), bins=range(0, 13), color='mediumpurple', alpha=0.8)
axes[1].set_title('Total Goals per Match')

result = matches_df.copy()
result['result'] = result.apply(
    lambda r: 'Draw' if r['winner'] == 0
    else ('Home Win' if r['winner'] == r['homeTeamId'] else 'Away Win'), axis=1
)
result['result'].value_counts().plot.pie(ax=axes[2], autopct='%1.1f%%',
    colors=['#4CAF50','#2196F3','#FF9800'], startangle=90)
axes[2].set_ylabel('')
axes[2].set_title('Match Outcomes')

plt.tight_layout()
plt.savefig(OUT / 'eda_match_outcomes.png', dpi=130)
plt.show()

In [ ]:
# ── Player role distribution ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
players_df['role'].value_counts().plot.bar(ax=ax, color='teal', alpha=0.8)
ax.set_title('Players by Position')
ax.set_xlabel('Role'); ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT / 'eda_player_roles.png', dpi=130)
plt.show()

---
## 3. Preprocess

In [ ]:
from src.data.preprocessor import preprocess, filter_passes

events_clean = preprocess(events_df, matches_df)
print(f'Events after cleaning: {len(events_clean):,}  (dropped {len(events_df)-len(events_clean):,})')

# Pass accuracy per competition
passes_all = filter_passes(events_clean, accurate_only=False)
acc = passes_all.groupby('competition')['accurate'].mean().sort_values(ascending=False)
acc.plot.bar(color='steelblue', figsize=(8, 4))
plt.title('Pass Accuracy by Competition'); plt.ylabel('Accuracy'); plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUT / 'eda_pass_accuracy.png', dpi=130)
plt.show()

---
## 4. Event Feature Extraction

In [ ]:
from src.features.event_features import extract_event_features

event_feats = extract_event_features(events_clean)
print(f'Feature matrix: {event_feats.shape}')
event_feats.describe().round(3)

In [ ]:
# ── Feature correlation heatmap ───────────────────────────────────────────────
feat_cols = [c for c in event_feats.columns if c not in {'matchId','teamId'}]
corr = event_feats[feat_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size':6},
            square=True, linewidths=0.3, ax=ax)
ax.set_title('Event Feature Correlations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'feature_correlation.png', dpi=130)
plt.show()

---
## 5. Passing Network Analysis

In [ ]:
from src.network.builder import build_all_networks
from src.network.metrics import compute_network_metrics

passes      = filter_passes(events_clean, accurate_only=True)
networks    = build_all_networks(passes)
net_metrics = compute_network_metrics(networks)

print(f'Networks: {len(networks):,}   Metric rows: {len(net_metrics):,}')
net_metrics.describe().round(3)

In [ ]:
# ── Sample network visualisation (top team by pass volume in first match) ──────
top_key = passes.groupby(['matchId','teamId']).size().idxmax()
G = networks[top_key]
team_name = teams_df.set_index('teamId')['name'].get(int(top_key[1]), str(top_key[1]))

weights  = [G[u][v]['weight'] for u, v in G.edges()]
node_deg = dict(G.degree(weight='weight'))
node_sizes = [node_deg.get(n, 1) * 15 for n in G.nodes()]
labels   = {n: players_df.set_index('playerId')['shortName'].get(n, str(n)) for n in G.nodes()}

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42, k=1.5)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='#FF6B35', alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, width=[w*0.25 for w in weights],
                       edge_color='steelblue', alpha=0.6, arrows=True,
                       arrowsize=10, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=6, ax=ax)
ax.set_title(f'Passing Network — {team_name}  (match {top_key[0]})', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(OUT / 'sample_passing_network.png', dpi=130)
plt.show()

In [ ]:
# ── Network metrics distribution ──────────────────────────────────────────────
net_plot_cols = ['density','clustering_coef','centralization',
                 'reciprocity','avg_shortest_path','top_player_share']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, net_plot_cols):
    net_metrics[col].dropna().hist(bins=40, ax=ax, color='mediumpurple', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Network Metric Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'network_metric_distributions.png', dpi=130)
plt.show()

---
## 6. Team Profile Aggregation

In [ ]:
from src.features.aggregator import aggregate_team_profiles, build_fingerprint_matrix

combined = event_feats.merge(net_metrics, on=['matchId','teamId'], how='inner')
profiles = aggregate_team_profiles(combined, matches_df)
profiles = profiles.merge(teams_df[['teamId','name']], on='teamId', how='left')

print(f'Team profiles: {len(profiles)}')

X, y = build_fingerprint_matrix(profiles)
feature_cols = [c for c in X.columns if c not in {'teamId','competition'}]

In [ ]:
# ── Top teams by win rate per competition ─────────────────────────────────────
top_teams = (
    profiles.merge(y[['teamId','competition','win_rate','points_per_match','goalDiff']],
                   on=['teamId','competition'], how='left')
    .sort_values('win_rate', ascending=False)
    .groupby('competition').head(3)[['competition','name','win_rate','points_per_match','goalDiff',
                                      'pass_accuracy','density','centralization']]
    .round(3)
)
print('Top 3 teams per competition by win rate')
display(top_teams)

In [ ]:
# ── Feature vs win rate scatter grid ─────────────────────────────────────────
full_profiles = profiles.merge(
    y[['teamId','competition','win_rate','points_per_match','goalDiff']],
    on=['teamId','competition'], how='left'
)

key_features = ['pass_accuracy','long_ball_ratio','cross_ratio',
                'density','centralization','pass_att_third','counter_rate','smart_pass_ratio']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flat, key_features):
    ax.scatter(full_profiles[feat], full_profiles['win_rate'],
               alpha=0.4, s=20, color='steelblue')
    # trend line
    valid = full_profiles[[feat,'win_rate']].dropna()
    if len(valid) > 5:
        z = np.polyfit(valid[feat], valid['win_rate'], 1)
        p = np.poly1d(z)
        xs = np.linspace(valid[feat].min(), valid[feat].max(), 100)
        ax.plot(xs, p(xs), color='red', lw=1.5, alpha=0.8)
        corr = valid[feat].corr(valid['win_rate'])
        ax.set_title(f'{feat}\nr={corr:.2f}', fontsize=8)
    ax.set_xlabel(''); ax.set_ylabel('Win Rate' if ax in axes[:,0] else '')

plt.suptitle('Feature vs Win Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'feature_vs_win_rate.png', dpi=130)
plt.show()

---
## 7. PCA Analysis

In [ ]:
from src.clustering.fingerprints import scale_features, run_pca, find_optimal_clusters
from src.visualization.plots import plot_pca_variance, plot_silhouette_curve

X_raw            = full_profiles[feature_cols].fillna(0).values
X_scaled, scaler = scale_features(X_raw)
X_pca, pca       = run_pca(X_scaled, n_components=min(10, len(feature_cols)))

plot_pca_variance(np.cumsum(pca.explained_variance_ratio_))
plt.savefig(OUT / 'pca_variance.png', dpi=130)
plt.show()

In [ ]:
# ── Top feature loadings for each PC ─────────────────────────────────────────
n_show = 3
loading_df = pd.DataFrame(pca.components_[:n_show].T,
                           index=feature_cols,
                           columns=[f'PC{i+1}' for i in range(n_show)])

fig, axes = plt.subplots(1, n_show, figsize=(5*n_show, 5))
for ax, pc in zip(axes, loading_df.columns):
    loading_df[pc].abs().sort_values(ascending=True).tail(10).plot.barh(ax=ax, color='teal', alpha=0.8)
    ax.set_title(f'{pc} — top feature loadings', fontsize=10)
plt.tight_layout()
plt.savefig(OUT / 'pca_loadings.png', dpi=130)
plt.show()

In [ ]:
# ── Optimal cluster count ─────────────────────────────────────────────────────
eval_df = find_optimal_clusters(X_pca, k_range=range(2, 10))
plot_silhouette_curve(eval_df)
plt.savefig(OUT / 'cluster_quality.png', dpi=130)
plt.show()

best_k = eval_df.loc[eval_df['silhouette'].idxmax(), 'k']
print(f'Best k by silhouette: {best_k}  (configured: {N_CLUSTERS})')
print(eval_df.to_string(index=False))

---
## 8. Tactical Fingerprints

In [ ]:
from src.clustering.fingerprints import build_fingerprint_report
from src.visualization.plots import (
    plot_cluster_scatter, plot_fingerprint_radar, plot_outcome_by_fingerprint
)

result   = build_fingerprint_report(
    profiles=full_profiles,
    feature_cols=feature_cols,
    n_clusters=N_CLUSTERS,
    use_umap=True,
)
labelled = result['profiles_labelled']
labelled = labelled.merge(teams_df[['teamId','name']], on='teamId', how='left')

plot_cluster_scatter(labelled, teams_df=teams_df, title='Tactical Fingerprints (UMAP)')
plt.savefig(OUT / 'fingerprint_scatter.png', dpi=130)
plt.show()

In [ ]:
# ── Radar chart — fingerprint style profiles ──────────────────────────────────
radar_features = ['pass_accuracy','long_ball_ratio','cross_ratio','smart_pass_ratio',
                  'density','centralization','pass_att_third','counter_rate']
plot_fingerprint_radar(labelled, feature_cols=radar_features)
plt.savefig(OUT / 'fingerprint_radar.png', dpi=130, bbox_inches='tight')
plt.show()

---
## 9. Fingerprints vs Match Outcomes

In [ ]:
for metric in ['win_rate', 'points_per_match', 'goalDiff']:
    plot_outcome_by_fingerprint(labelled, metric=metric)
    plt.savefig(OUT / f'{metric}_by_fp.png', dpi=130)
    plt.show()

In [ ]:
# ── Feature means per fingerprint — heatmap ───────────────────────────────────
fp_means = labelled.groupby('fingerprint')[feature_cols].mean()
fp_norm  = (fp_means - fp_means.min()) / (fp_means.max() - fp_means.min() + 1e-9)

fig, ax = plt.subplots(figsize=(16, max(4, N_CLUSTERS*0.8)))
sns.heatmap(fp_norm.T, cmap='RdYlGn', center=0.5,
            annot=True, fmt='.2f', annot_kws={'size':7},
            linewidths=0.3, ax=ax)
ax.set_title('Normalised Feature Means by Tactical Fingerprint',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Fingerprint')
plt.tight_layout()
plt.savefig(OUT / 'fingerprint_feature_heatmap.png', dpi=130)
plt.show()

---
## 10. Cross-Competition Analysis

In [ ]:
# Fingerprint distribution per competition
fp_comp = labelled.groupby(['competition','fingerprint']).size().unstack(fill_value=0)
fp_comp_pct = fp_comp.div(fp_comp.sum(axis=1), axis=0)

fp_comp_pct.plot.bar(stacked=True, figsize=(10,5), colormap='tab10')
plt.title('Fingerprint Distribution by Competition', fontsize=12, fontweight='bold')
plt.xlabel('Competition'); plt.ylabel('Proportion of teams')
plt.xticks(rotation=20); plt.legend(title='Fingerprint', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.savefig(OUT / 'fingerprint_by_competition.png', dpi=130)
plt.show()

In [ ]:
# Tactical style comparison across competitions
style_cols = ['pass_accuracy','long_ball_ratio','cross_ratio','smart_pass_ratio',
              'counter_rate','pass_att_third']
comp_style = labelled.groupby('competition')[style_cols].mean()

fig, ax = plt.subplots(figsize=(12, 5))
comp_style.plot.bar(ax=ax, colormap='tab10', alpha=0.85)
ax.set_title('Tactical Style by Competition', fontsize=12, fontweight='bold')
ax.set_xlabel('Competition'); ax.set_ylabel('Mean value')
ax.legend(bbox_to_anchor=(1,1))
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUT / 'style_by_competition.png', dpi=130)
plt.show()

---
## 11. Summary Table & Save

In [ ]:
summary = labelled.groupby('fingerprint').agg(
    n_teams            = ('teamId', 'nunique'),
    win_rate           = ('win_rate', 'mean'),
    points_per_match   = ('points_per_match', 'mean'),
    goal_diff          = ('goalDiff', 'mean'),
    pass_accuracy      = ('pass_accuracy', 'mean'),
    long_ball_ratio    = ('long_ball_ratio', 'mean'),
    cross_ratio        = ('cross_ratio', 'mean'),
    smart_pass_ratio   = ('smart_pass_ratio', 'mean'),
    counter_rate       = ('counter_rate', 'mean'),
    density            = ('density', 'mean'),
    centralization     = ('centralization', 'mean'),
    clustering_coef    = ('clustering_coef', 'mean'),
).round(3)

print('\n── Tactical Fingerprint Summary ──')
display(summary)

In [ ]:
# Teams per fingerprint
for fp_id, grp in labelled.groupby('fingerprint'):
    teams_list = ', '.join(sorted(grp['name'].dropna().unique()))
    print(f'\n── FP {fp_id}  ({len(grp)} teams) ──')
    print(teams_list)

In [ ]:
# Save everything to Drive
labelled.to_csv(OUT / 'fingerprints.csv', index=False)
labelled.to_parquet(OUT / 'fingerprints.parquet', index=False)
summary.to_csv(OUT / 'fingerprint_summary.csv')

import pickle
with open(OUT / 'pipeline_artifacts.pkl', 'wb') as f:
    pickle.dump({'scaler': result['scaler'], 'pca': result['pca']}, f)

print(f'\nAll results saved to {DRIVE_OUT}')
print('Files written:')
for p in sorted(OUT.glob('*')):
    print(f'  {p.name}  ({p.stat().st_size/1024:.0f} KB)')